In [1]:
from os import listdir
import os.path as op
import numpy as np

bids_folder = '/data/ds-stressrisk'

subList = ['01','02','03','04','05','09']
subList = [f[4:6] for f in listdir(bids_folder) if f[0:4] == 'sub-']
ses = 1

target_folder = op.join(bids_folder,'derivatives','correlation_matrices')


In [2]:
from nilearn import datasets

atlas = datasets.fetch_atlas_surf_destrieux()
regions = atlas['labels'].copy()
masked_regions = [b'Medial_wall', b'Unknown']
masked_labels = [regions.index(r) for r in masked_regions] # [42, 0]

# Build Destrieux parcellation and mask
labeling = np.concatenate([atlas['map_left'], atlas['map_right']]) # atlas['map_left'] == atlas.map_left -> array, each vertex has a label assignment (a number from 0-51)
mask = ~np.isin(labeling, masked_labels)
N_vertices = len(np.where(mask==True)[0])

In [5]:
#matrix_zeros = np.zeros((20484, 20484))
matrix_zeros = np.zeros((N_vertices, N_vertices))
av_cm = matrix_zeros.copy()

for sub in subList:
    #correlation_matrix_resize = np.load(op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_fsav5-format.npy'))
    try:
        correlation_matrix = np.load(op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_fsav5_unfiltered.npy'))
        av_cm += np.arctan(correlation_matrix) # fisher-Z-transformed
        print(f'subject {sub} added')
    except:
        print(f'subject {sub} failed')

av_cm = av_cm/len(subList)
av_cm_transf = np.tan(av_cm) # sanity check: diagonal should be 1 !

subject 12 added
subject 17 added
subject 18 added
subject 61 added
subject 22 added
subject 21 added
subject 34 added
subject 40 added
subject 14 added
subject 09 added
subject 57 added
subject 41 added
subject 03 added
subject 50 added
subject 04 added
subject 44 added
subject 13 added
subject 51 added
subject 26 added
subject 59 added
subject 48 added
subject 19 added
subject 52 added
subject 46 added
subject 10 added
subject 23 added
subject 39 added
subject 31 added
subject 25 added
subject 42 added
subject 05 added
subject 53 added
subject 28 added
subject 47 added
subject 54 added
subject 45 added
subject 35 added
subject 36 added
subject 16 added
subject 01 added
subject 32 added
subject 58 added
subject 37 added
subject 43 added
subject 49 added
subject 55 added
subject 38 added
subject 24 added
subject 30 added
subject 02 added


In [6]:
from brainspace.gradient import GradientMaps

gm = GradientMaps(n_components=3, random_state=0) # Default is 'dm' = DiffusionMaps
gm.fit(av_cm_transf)

file_name = f'cm_av{len(subList)}_unfiltered.npy'
np.save(op.join(target_folder,file_name),gm.gradients_)
print(f'saved to: {op.join(target_folder,file_name)}')

/home/ubuntu/miniconda3/envs/numrefields/lib/python3.10/site-packages/brainspace-0.1.4-py3.10.egg/brainspace/gradient/embedding.py:70: UserWarning: Affinity is not symmetric. Making symmetric.
  warnings.warn('Affinity is not symmetric. Making symmetric.')


saved to: /data/ds-stressrisk/derivatives/correlation_matrices/cm_av50_unfiltered.npy


In [2]:
# inspect CMs

sub = subList[0]
correlation_matrix = np.load(op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_fsav5_unfiltered.npy'))


In [7]:
import matplotlib.pyplot as plt

plt.hist(correlation_matrix.flatten(), bins=100)
plt.xlim(-1,1)
plt.title(f'subject {sub}')
plt.savefig(op.join(target_folder,'plots',f'sub-{sub}.pdf'))